# Step 1: Dataset, Preprocessing & Data Quality Audit

**Governance Auditing Framework for AI-Based Social Welfare Eligibility Decision Systems**

This notebook covers the first stage of the pipeline described in the survey paper (Section 5.2, *Input Data and Data Quality Assessment*):

1. Load the Folktables **ACS Income** dataset (Phase 1 technical-validation dataset)
2. Understand the data (shape, types, class balance, group distribution)
3. **Audit data quality BEFORE cleaning** — Missing Value Rate, Duplicate Rate, Completeness (paper eqns 1–3)
4. Preprocess/clean the data
5. Audit data quality AFTER cleaning, to show what the cleaning step actually fixed
6. Produce the train/test split (with the protected attribute kept aligned) that later modules (fairness, explainability, reliability, robustness) will reuse

Fairness, explainability, reliability, robustness, compliance and the Governance Risk Score are **out of scope for this notebook** — they come later, once a baseline model exists.

## 0. Setup

The notebook lives in `notebooks/`, but the reusable code lives in `src/` at the project root, so we add the project root to `sys.path`.

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

from src.data_pipeline import (
    load_acs_income,
    build_income_task,
    clean_features,
    train_test_split_with_group,
    PROTECTED_ATTRIBUTE,
)
from src.audit.data_quality import audit_data_quality

pd.set_option("display.max_columns", 50)

## 1. Load the raw ACS PUMS data

`load_acs_income()` downloads one state-year of US Census American Community Survey (ACS) Public Use Microdata (PUMS) via the `folktables` package. **The first run needs internet access**; folktables caches the CSV afterwards under its own data directory, so later runs are offline and fast.

We use California, **2022**, 1-year person-level data, the most recent 1-Year ACS extract available. Note: the Census Bureau renamed the "relationship to householder" column from `RELP` to `RELSHIPP` starting with the 2019 data dictionary. `folktables`' built-in `ACSIncome` task still hardcodes `RELP` and breaks on 2019+ data, so `build_income_task()` (in `src/data_pipeline.py`) reconstructs the same task definition with the year-appropriate column name instead of using the built-in task directly.

In [2]:
DATA_YEAR = "2022"

acs_data = load_acs_income(state="CA", year=DATA_YEAR)
print("Raw ACS extract shape:", acs_data.shape)
acs_data.head()

Raw ACS extract shape: (391171, 287)


,RT,SERIALNO,DIVISION,SPORDER,PUMA,REGION,ST,ADJINC,PWGTP,AGEP,CIT,CITWP,COW,DDRS,DEAR,DEYE,DOUT,DPHY,DRAT,DRATX,DREM,ENG,FER,GCL,GCM,...,PWGTP56,PWGTP57,PWGTP58,PWGTP59,PWGTP60,PWGTP61,PWGTP62,PWGTP63,PWGTP64,PWGTP65,PWGTP66,PWGTP67,PWGTP68,PWGTP69,PWGTP70,PWGTP71,PWGTP72,PWGTP73,PWGTP74,PWGTP75,PWGTP76,PWGTP77,PWGTP78,PWGTP79,PWGTP80
0,P,2022GQ0000033,9,1,6509,4,6,1042311,14,56,1,NaN,NaN,2.0,2,2,1.0,2.0,NaN,NaN,1.0,NaN,NaN,2.0,NaN,...,14,13,12,14,12,13,13,12,12,12,12,12,12,12,11,12,13,13,11,14,12,11,11,13,10
1,P,2022GQ0000039,9,1,6501,4,6,1042311,27,52,5,NaN,NaN,2.0,2,1,2.0,1.0,NaN,NaN,2.0,3.0,NaN,2.0,NaN,...,27,6,26,29,4,6,6,28,27,29,27,48,5,27,29,27,46,6,25,27,48,27,43,47,25
2,P,2022GQ0000041,9,1,8101,4,6,1042311,70,61,1,NaN,NaN,1.0,2,2,2.0,1.0,5.0,1.0,2.0,NaN,NaN,2.0,NaN,...,58,71,18,58,90,79,90,90,90,70,69,70,71,20,12,10,57,78,91,20,58,71,20,56,12
3,P,2022GQ0000067,9,1,8303,4,6,1042311,22,26,4,2020.0,1.0,2.0,2,2,2.0,2.0,NaN,NaN,1.0,1.0,2.0,NaN,NaN,...,25,37,29,18,2,4,3,29,18,27,28,34,2,26,23,22,38,5,19,21,35,25,56,30,24
4,P,2022GQ0000070,9,1,6712,4,6,1042311,8,38,1,NaN,5.0,2.0,2,2,2.0,2.0,NaN,NaN,2.0,NaN,2.0,2.0,NaN,...,15,16,3,2,2,8,15,8,8,8,1,1,9,2,16,3,15,8,14,17,9,8,8,8,3


## 2. Apply the ACSIncome task definition

`build_income_task()` reconstructs folktables' `ACSIncome` task (same features, target, and row filters), converting the raw survey extract into:

- **features**: AGEP, COW, SCHL, MAR, OCCP, POBP, **RELSHIPP** (relationship to householder, replacing RELP for 2019+ data), WKHP, SEX, RAC1P
- **label**: PINCP > \$50,000 (binary), used here as a **proxy for "eligibility"** for Phase 1 technical validation. It is *not* a real welfare-eligibility label; that distinction matters when discussing limitations in your report.
- **group**: RAC1P, the protected attribute used later by the Fairness module

folktables also applies ACSIncome's built-in row filters internally (age > 16, usual hours worked > 0, reported income > 100).

In [3]:
features, label, group = build_income_task(acs_data, year=DATA_YEAR)
print("Features shape:", features.shape)
print("Columns:", list(features.columns))
features.head()

Features shape: (200577, 10)
Columns: ['AGEP', 'COW', 'SCHL', 'MAR', 'OCCP', 'POBP', 'RELSHIPP', 'WKHP', 'SEX', 'RAC1P']


,AGEP,COW,SCHL,MAR,OCCP,POBP,RELSHIPP,WKHP,SEX,RAC1P
0,26.0,1.0,19.0,5.0,310.0,303.0,38.0,30.0,2.0,8.0
1,38.0,5.0,16.0,3.0,2910.0,4.0,37.0,40.0,2.0,1.0
2,23.0,5.0,16.0,5.0,9825.0,30.0,38.0,40.0,1.0,9.0
3,20.0,1.0,19.0,5.0,4055.0,32.0,38.0,20.0,1.0,8.0
4,20.0,5.0,16.0,5.0,9830.0,47.0,38.0,50.0,2.0,9.0


## 3. Understand the data (EDA)

Before auditing quality formally, get a feel for the data: types, ranges, class balance, and the distribution of the protected attribute. This matters because a governance audit later needs to explain *why* a metric looks the way it does — e.g. a small group in `RAC1P` will produce noisier fairness estimates regardless of the model.

In [4]:
features.info()

<class 'pandas.DataFrame'>
RangeIndex: 200577 entries, 0 to 200576
Data columns (total 10 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   AGEP      200577 non-null  float64
 1   COW       200577 non-null  float64
 2   SCHL      200577 non-null  float64
 3   MAR       200577 non-null  float64
 4   OCCP      200577 non-null  float64
 5   POBP      200577 non-null  float64
 6   RELSHIPP  200577 non-null  float64
 7   WKHP      200577 non-null  float64
 8   SEX       200577 non-null  float64
 9   RAC1P     200577 non-null  float64
dtypes: float64(10)
memory usage: 15.3 MB


In [5]:
features.describe()

,AGEP,COW,SCHL,MAR,OCCP,POBP,RELSHIPP,WKHP,SEX,RAC1P
count,200577.000000,200577.000000,200577.000000,200577.000000,200577.000000,200577.000000,200577.000000,200577.000000,200577.000000,200577.000000
mean,42.863324,2.179592,18.508299,2.728982,3890.997034,95.314303,23.032257,37.550582,1.471619,4.559356
std,15.020499,1.889448,4.142823,1.865814,2673.715719,123.800735,4.857293,12.943703,0.499195,3.354286
min,17.000000,1.000000,1.000000,1.000000,10.000000,1.000000,20.000000,1.000000,1.000000,1.000000
25%,30.000000,1.000000,16.000000,1.000000,1555.000000,6.000000,20.000000,34.000000,1.000000,1.000000
50%,42.000000,1.000000,19.000000,1.000000,3930.000000,6.000000,21.000000,40.000000,1.000000,6.000000
75%,55.000000,3.000000,21.000000,5.000000,5400.000000,212.000000,25.000000,40.000000,2.000000,8.000000
max,94.000000,8.000000,24.000000,5.000000,9830.000000,554.000000,38.000000,99.000000,2.000000,9.000000


In [6]:
print("Label (eligibility proxy) balance:")
print(label.value_counts(normalize=True).rename("proportion"))

print("\nProtected attribute (RAC1P) distribution:")
print(group.value_counts())

Label (eligibility proxy) balance:
label
False    0.526735
True     0.473265
Name: proportion, dtype: float64

Protected attribute (RAC1P) distribution:
RAC1P
1    83094
6    37660
8    34290
9    33151
2     8422
3     2676
7      743
5      532
4        9
Name: count, dtype: int64


**What to look for here:**
- Is the label roughly balanced, or skewed? A skewed label affects which fairness/reliability metrics are informative later.
- Are some `RAC1P` groups very small? Small groups make demographic-parity/equal-opportunity estimates unstable — worth noting as a limitation.
- Do any numeric columns (e.g. `WKHP`, `AGEP`) have implausible values (0s, negative, extreme outliers)? These often hide as "valid but wrong" values that a pure missing-value check won't catch.

### 3a. Demographic & Protected-Attribute Understanding

This step matters for two later modules directly:

- **Fairness module**: needs the label rate per group (base rate) as context before computing Demographic Parity, Disparate Impact, and Equal Opportunity Difference. A large base-rate gap between groups is itself a signal worth reporting, independent of the model.
- **Explainability module**: needs to know how many observations exist per group. A model can look unbiased on a tiny subgroup simply because there isn't enough data to detect a real effect.

We check `RAC1P` (the paper's chosen protected attribute) and `SEX` (commonly audited alongside race in this dataset).

In [7]:
label_by_group = pd.concat([
    features["RAC1P"].rename("RAC1P"),
    features["SEX"].rename("SEX"),
    label.rename("label"),
], axis=1)

print("Eligibility-proxy rate by RAC1P group:")
print(label_by_group.groupby("RAC1P")["label"].agg(["mean", "count"]).sort_values("mean", ascending=False))

print("\nEligibility-proxy rate by SEX group:")
print(label_by_group.groupby("SEX")["label"].agg(["mean", "count"]))

Eligibility-proxy rate by RAC1P group:


           mean  count
RAC1P                 
1.0    0.572243  83094
6.0    0.552735  37660
4.0    0.444444      9
2.0    0.415816   8422
7.0    0.403769    743
9.0    0.381678  33151
3.0    0.324365   2676
5.0    0.308271    532
8.0    0.264479  34290

Eligibility-proxy rate by SEX group:
         mean   count
SEX                  
1.0  0.518904  105981
2.0  0.422132   94596


**How to read this:** the `mean` column is the positive-outcome rate for that group, comparable to what the Fairness module's Demographic Parity metric will later compute on model predictions instead of ground truth. If the ground-truth rates already differ a lot between groups, expect Demographic Parity Difference to be nonzero even for a well-behaved model, because the underlying population isn't balanced. That distinction, a data-driven gap versus a model-introduced gap, is worth stating explicitly in your report.

In [8]:
intersectional = pd.crosstab(features["RAC1P"], features["SEX"])
print("RAC1P x SEX group sizes:")
print(intersectional)

small_cells = intersectional[intersectional < 30].stack().dropna()
if small_cells.empty:
    print("\nNo RAC1P x SEX subgroup has fewer than 30 records.")
else:
    print(f"\n{len(small_cells)} RAC1P x SEX subgroup(s) have fewer than 30 records "
          "(fairness metrics computed on these will be unstable):")
    print(small_cells)

RAC1P x SEX group sizes:
SEX      1.0    2.0
RAC1P              
1.0    44421  38673
2.0     4186   4236
3.0     1470   1206
4.0        5      4
5.0      268    264
6.0    18729  18931
7.0      382    361
8.0    18850  15440
9.0    17670  15481



2 RAC1P x SEX subgroup(s) have fewer than 30 records (fairness metrics computed on these will be unstable):
RAC1P  SEX
4.0    1.0    5.0
       2.0    4.0
dtype: float64


### 3b. Feature Types, Cardinality & Validity Checks

This matters for preprocessing choices and for the Explainability module:

- **High-cardinality categorical features** (`OCCP` has several hundred occupation codes, `POBP` has roughly 200 birthplace codes) need a deliberate encoding decision. One-hot encoding them directly would blow up dimensionality and would also fragment SHAP importance across hundreds of dummy columns, making the explainability output hard to read. XGBoost's native categorical support, or a target/frequency encoding, is usually a better fit here than plain one-hot.
- **Validity** (are values inside a plausible range) is a data-quality dimension the Missing-Value/Duplicate/Completeness formulas do not catch. A row with `WKHP = 0` is not missing, but it is still wrong for someone who is supposedly employed.

In [9]:
cardinality = features.nunique().sort_values(ascending=False)
print("Unique values per column:")
print(cardinality)

high_cardinality_cols = cardinality[cardinality > 20].index.tolist()
low_cardinality_cols = cardinality[cardinality <= 20].index.tolist()
print("\nHigh-cardinality columns (need an encoding strategy beyond plain one-hot):", high_cardinality_cols)
print("Low-cardinality columns (safe for one-hot if needed):", low_cardinality_cols)

Unique values per column:
OCCP        528
POBP        221
WKHP         97
AGEP         75
SCHL         24
RELSHIPP     19
RAC1P         9
COW           8
MAR           5
SEX           2
dtype: int64



High-cardinality columns (need an encoding strategy beyond plain one-hot): ['OCCP', 'POBP', 'WKHP', 'AGEP', 'SCHL']
Low-cardinality columns (safe for one-hot if needed): ['RELSHIPP', 'RAC1P', 'COW', 'MAR', 'SEX']


In [10]:
numeric_cols = ["AGEP", "WKHP"]
for col in numeric_cols:
    print(f"{col}: min={features[col].min()}, max={features[col].max()}, "
          f"mean={features[col].mean():.1f}")

implausible_hours = features[(features["WKHP"] <= 0) | (features["WKHP"] > 99)]
print(f"\nRows with implausible WKHP (<=0 or >99): {len(implausible_hours)}")

AGEP: min=17.0, max=94.0, mean=42.9
WKHP: min=1.0, max=99.0, mean=37.6

Rows with implausible WKHP (<=0 or >99): 0


### 3c. Protected Attributes as Proxies in Other Features

Even though `RAC1P` and `SEX` are the explicit protected attributes, other features can correlate with them and act as indirect proxies. For example, `POBP` (birthplace) correlates with national origin, and `OCCP` (occupation) can correlate with both sex and race due to historical labor-market patterns. This matters for the Explainability module's Sensitive Feature Influence metric (survey eqn 9): if a supposedly neutral feature turns out to carry most of a protected attribute's signal, a model can look fair on RAC1P/SEX metrics while still discriminating through that proxy.

A quick way to check this now, before any model exists, is to compare feature distributions across groups.

In [11]:
print("Mean AGEP and WKHP by RAC1P group:")
print(features.groupby("RAC1P")[["AGEP", "WKHP"]].mean())

print("\nEducation level (SCHL) distribution by RAC1P group (top 5 levels, proportion within group):")
schl_by_group = pd.crosstab(features["RAC1P"], features["SCHL"], normalize="index")
top_schl_cols = features["SCHL"].value_counts().head(5).index
print(schl_by_group[top_schl_cols])

Mean AGEP and WKHP by RAC1P group:


            AGEP       WKHP
RAC1P                      
1.0    45.157220  37.842516
2.0    43.287461  37.582403
3.0    40.750000  37.308296
4.0    42.222222  37.000000
5.0    39.922932  37.077068
6.0    43.071057  37.326500
7.0    42.057873  38.432032
8.0    39.645290  37.422689
9.0    40.334470  37.205152

Education level (SCHL) distribution by RAC1P group (top 5 levels, proportion within group):


SCHL       21.0      16.0      19.0      22.0      20.0
RAC1P                                                  
1.0    0.297242  0.137699  0.143681  0.132729  0.082340
2.0    0.204346  0.204821  0.193303  0.104607  0.096414
3.0    0.125187  0.227952  0.174514  0.056054  0.077728
4.0    0.111111  0.333333  0.111111  0.111111  0.000000
5.0    0.112782  0.266917  0.152256  0.048872  0.078947
6.0    0.357833  0.107329  0.101036  0.170074  0.066808
7.0    0.166891  0.267833  0.166891  0.060565  0.096904
8.0    0.111344  0.271741  0.141820  0.030417  0.065646
9.0    0.188682  0.209104  0.160478  0.071189  0.078610


### 3d. Defining the Sensitive-Feature Set for Later Modules

The Fairness module needs one or more **protected attributes**, used to slice predictions into groups. The Explainability module separately needs a **sensitive-feature list**, used to compute how much each sensitive feature contributes to predictions via SHAP. These lists overlap but are not identical: a protected attribute is always sensitive, but a proxy feature identified in the previous step, like `POBP`, can be flagged as sensitive for the explainability check even though the Fairness module still slices by `RAC1P` directly.

Recording this decision now, at the preprocessing stage, means the Explainability module built later can just import this list instead of re-deriving it.

In [12]:
PROTECTED_ATTRIBUTES = ["RAC1P", "SEX"]
SENSITIVE_FEATURES = ["RAC1P", "SEX", "POBP"]  # POBP included as a proxy candidate identified above

print("Protected attributes (Fairness module):", PROTECTED_ATTRIBUTES)
print("Sensitive features (Explainability module):", SENSITIVE_FEATURES)

Protected attributes (Fairness module): ['RAC1P', 'SEX']
Sensitive features (Explainability module): ['RAC1P', 'SEX', 'POBP']


## Summary & next steps

- Loaded Folktables ACS Income (CA, 2018) as the Phase 1 technical-validation dataset.
- Examined demographic base rates, intersectional group sizes, feature cardinality, validity ranges, and protected-attribute proxies (RAC1P, SEX, POBP), and recorded `PROTECTED_ATTRIBUTES` and `SENSITIVE_FEATURES` for direct reuse by the Fairness and Explainability modules.
- Measured data quality **before** cleaning using the paper's own formulas (§4.1), so the numbers are audit evidence, not an assumption.
- Cleaned the data (drop duplicates, drop nulls) and re-measured to document the effect.
- Produced a stratified train/test split with the protected attributes preserved for the Fairness module.
- Saved the processed split to `data/processed/` for reuse.

**Next (separate notebook, later):** train the baseline XGBoost eligibility classifier on `data/processed/train.csv`, evaluate on `test.csv`, then move on to the Fairness, Explainability, Reliability and Robustness audit modules.

## 4. Data Quality Audit — BEFORE cleaning

This runs the exact formulas from survey §4.1:

- **Missing Value Rate** = missing observations / total observations
- **Duplicate Rate** = duplicate records / total records
- **Data Completeness** = 1 − (total missing values / total observations)

It's important this runs on the **raw** features, before any cleaning — auditing already-cleaned data would hide exactly the issues the audit exists to catch.

In [13]:
report_before = audit_data_quality(features)
print(report_before.summary())

Rows x Columns:            200577 x 10
Missing Value Rate (overall): 0.0000
Duplicate Rate:               0.0367
Data Completeness:            1.0000
Data Quality Risk Score:      0.0184  (0 = clean, 1 = worst)

Missing Value Rate by column (non-zero only):
  (none)


## 5. Preprocessing / Cleaning

`clean_features()` currently does the minimal cleaning needed to train a baseline model:

1. **Drop duplicate rows** — duplicates would otherwise give some observations extra influence during training and distort evaluation metrics.
2. **Drop rows with missing values** — XGBoost can technically handle NaNs natively, but for Phase 1 we keep preprocessing simple and explicit; missingness handling strategy (drop vs. impute) is worth revisiting once you see how much data it costs.

After cleaning, `label` and `group` must be re-aligned to the surviving row index — they were built from the same original index as `features`, so dropped rows must be dropped from all three together.

In [14]:
cleaned_features = clean_features(features)
cleaned_label = label.loc[cleaned_features.index]
cleaned_group = group.loc[cleaned_features.index]

print(f"Rows before cleaning: {features.shape[0]}")
print(f"Rows after cleaning:  {cleaned_features.shape[0]}")
print(f"Rows dropped:         {features.shape[0] - cleaned_features.shape[0]} "
      f"({(features.shape[0] - cleaned_features.shape[0]) / features.shape[0]:.2%})")

Rows before cleaning: 200577
Rows after cleaning:  193208
Rows dropped:         7369 (3.67%)


## 6. Data Quality Audit — AFTER cleaning

Re-running the same audit on the cleaned data should show Missing Value Rate → 0, Duplicate Rate → 0, and Completeness → 1, by construction. The point of showing both reports side by side is to **document what cleaning actually removed** — this becomes evidence in your report, not just an assumption.

In [15]:
report_after = audit_data_quality(cleaned_features)
print(report_after.summary())

Rows x Columns:            193208 x 10
Missing Value Rate (overall): 0.0000
Duplicate Rate:               0.0000
Data Completeness:            1.0000
Data Quality Risk Score:      0.0000  (0 = clean, 1 = worst)

Missing Value Rate by column (non-zero only):
  (none)


In [16]:
comparison = pd.DataFrame({
    "before_cleaning": {
        "missing_value_rate": report_before.missing_value_rate_overall,
        "duplicate_rate": report_before.duplicate_rate,
        "completeness": report_before.completeness,
        "risk_score": report_before.risk_score,
    },
    "after_cleaning": {
        "missing_value_rate": report_after.missing_value_rate_overall,
        "duplicate_rate": report_after.duplicate_rate,
        "completeness": report_after.completeness,
        "risk_score": report_after.risk_score,
    },
})
comparison

,before_cleaning,after_cleaning
missing_value_rate,0.000000,0.0
duplicate_rate,0.036739,0.0
completeness,1.000000,1.0
risk_score,0.018370,0.0


## 7. Train/test split (with protected attribute kept aligned)

This split is **stratified on the label** so both splits keep the same eligibility-proxy balance, and it keeps `group` (RAC1P) aligned by index so the Fairness module can later slice predictions by group without re-joining anything.

In [17]:
split = train_test_split_with_group(cleaned_features, cleaned_label, cleaned_group)

print("X_train:", split.X_train.shape, " X_test:", split.X_test.shape)
print("Train label balance:\n", split.y_train.value_counts(normalize=True))
print("\nTest label balance:\n", split.y_test.value_counts(normalize=True))

X_train: (154566, 10)  X_test: (38642, 10)
Train label balance:
 

label
False    0.521052
True     0.478948
Name: proportion, dtype: float64

Test label balance:
 label
False    0.521039
True     0.478961
Name: proportion, dtype: float64


## 8. Persist the processed split

Save to `data/processed/` so the next notebook (baseline XGBoost model) can load it directly instead of re-downloading and re-cleaning.

In [18]:
out_dir = os.path.abspath("../data/processed")
os.makedirs(out_dir, exist_ok=True)

split.X_train.assign(label=split.y_train, group=split.group_train).to_csv(
    os.path.join(out_dir, "train.csv"), index=False
)
split.X_test.assign(label=split.y_test, group=split.group_test).to_csv(
    os.path.join(out_dir, "test.csv"), index=False
)
print("Saved to", out_dir)

Saved to E:\MTECH\Final_yr_project\data\processed


## Summary & next steps

- Loaded Folktables ACS Income (CA, 2018) as the Phase 1 technical-validation dataset.
- Measured data quality **before** cleaning using the paper's own formulas (§4.1), so the numbers are audit evidence, not an assumption.
- Cleaned the data (drop duplicates, drop nulls) and re-measured to document the effect.
- Produced a stratified train/test split with the protected attribute (`RAC1P`) preserved for the Fairness module.
- Saved the processed split to `data/processed/` for reuse.

**Next (separate notebook, later):** train the baseline XGBoost eligibility classifier on `data/processed/train.csv`, evaluate on `test.csv`, then move on to the Fairness, Explainability, Reliability and Robustness audit modules.